# Angular Power Spectrum Pipeline: Tracers → C(ℓ) → Database → Latent Space

This notebook demonstrates the full pipeline for angular power spectrum
computations, storing all components in the database:

1. Store cosmological parameters in the DB
2. Store tracer configurations (weak lensing, number counts, CMB lensing)
3. Compute angular C(ℓ) with provenance tracking
4. Inspect the stored provenance
5. Visualize the power spectra
6. Encode C(ℓ) curves into a PCA latent space

> Uses `nest_asyncio` and a file-backed SQLite DB for Jupyter compatibility.

In [ ]:
import nest_asyncio
nest_asyncio.apply()

import asyncio
import numpy as np
import matplotlib.pyplot as plt

## 1. Initialize the database

In [ ]:
import tisserande.db
from tissage_cosmique.db.base import Base, init_db, get_session, close_db

DB_URL = "sqlite+aiosqlite:///angular_cl_demo.db"

asyncio.run(close_db())
init_db(DB_URL)

async def _setup():
    async with get_session() as session:
        conn = await session.connection()
        await conn.run_sync(Base.metadata.drop_all)
        await conn.run_sync(Base.metadata.create_all)

asyncio.run(_setup())

from tisserande.tracking.backends import LocalSyncBackend
from tisserande.tracking.decorator import configure
configure(backend=LocalSyncBackend())
print("Database initialized, tracking configured")

## 2. Store cosmological parameters

In [ ]:
from tissage_cosmique.local_sync import cosmology_params

planck = cosmology_params.create_row(
    name="Planck2018",
    Omega_c=0.2589, Omega_b=0.0486, h=0.6774,
    n_s=0.9667, sigma8=0.8159,
    Omega_k=0.0, w0=-1.0, wa=0.0,
)

wcdm = cosmology_params.create_row(
    name="wCDM",
    Omega_c=0.25, Omega_b=0.05, h=0.70,
    n_s=0.96, sigma8=0.80,
    Omega_k=0.0, w0=-0.9, wa=0.1,
)

print(f"Stored: {planck.name} (id={planck.id_})")
print(f"Stored: {wcdm.name} (id={wcdm.id_})")

## 3. Store tracer configurations

Define three tracer types: a weak lensing source sample, a galaxy
clustering lens sample, and CMB lensing.

In [ ]:
from tissage_cosmique.local_sync import tracer_config

# Redshift distributions
z = np.linspace(0.0, 3.0, 200)

# Source galaxies: peaked at z=0.8
nz_source = np.exp(-0.5 * ((z - 0.8) / 0.2)**2)

# Lens galaxies: peaked at z=0.4
nz_lens = np.exp(-0.5 * ((z - 0.4) / 0.15)**2)

# Weak lensing tracer
wl_tracer = tracer_config.create_row(
    name="source_bin1",
    tracer_type="weak_lensing",
    dndz_z=z.tolist(),
    dndz_nz=nz_source.tolist(),
    has_shear=True,
)
print(f"Stored: {wl_tracer.name} ({wl_tracer.tracer_type})")

# Number counts tracer with constant bias b=1.5
nc_tracer = tracer_config.create_row(
    name="lens_bin1",
    tracer_type="number_counts",
    dndz_z=z.tolist(),
    dndz_nz=nz_lens.tolist(),
    bias_z=z.tolist(),
    bias_bz=(np.ones_like(z) * 1.5).tolist(),
    has_rsd=False,
)
print(f"Stored: {nc_tracer.name} ({nc_tracer.tracer_type})")

# CMB lensing tracer
cmb_tracer = tracer_config.create_row(
    name="cmb_kappa",
    tracer_type="cmb_lensing",
    z_source=1100.0,
)
print(f"Stored: {cmb_tracer.name} ({cmb_tracer.tracer_type})")

In [ ]:
# Visualize the redshift distributions
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(z, nz_source / nz_source.max(), label="Source (WL)", lw=2)
ax.plot(z, nz_lens / nz_lens.max(), label="Lens (NC)", lw=2)
ax.set_xlabel("Redshift z")
ax.set_ylabel("Normalized N(z)")
ax.set_title("Tracer Redshift Distributions")
ax.legend()
ax.set_xlim(0, 2)
plt.tight_layout()
plt.show()

## 4. Compute angular C(ℓ) with provenance tracking

Each call to `angular_cl` is decorated with `@track`, so tisserande
records the cosmology params, tracer configs, and ℓ values as input
nodes, and the C(ℓ) array as the output node.

In [ ]:
from tissage_cosmique.computations.angular_cl import angular_cl

ell = np.arange(10, 2000).astype(float)

# Get configs as dicts (exclude id_ to avoid UUID JSON issues)
wl_dict = wl_tracer.model_dump(exclude={"id_"})
nc_dict = nc_tracer.model_dump(exclude={"id_"})
cmb_dict = cmb_tracer.model_dump(exclude={"id_"})
planck_dict = planck.model_dump(exclude={"id_"})
wcdm_dict = wcdm.model_dump(exclude={"id_"})

# Compute all six spectra for Planck cosmology
print("Computing C(ell) for Planck2018...")
cl_wl_wl = angular_cl(planck_dict, wl_dict, wl_dict, ell)
cl_nc_nc = angular_cl(planck_dict, nc_dict, nc_dict, ell)
cl_wl_nc = angular_cl(planck_dict, wl_dict, nc_dict, ell)
cl_cmb_cmb = angular_cl(planck_dict, cmb_dict, cmb_dict, ell)
cl_wl_cmb = angular_cl(planck_dict, wl_dict, cmb_dict, ell)
cl_nc_cmb = angular_cl(planck_dict, nc_dict, cmb_dict, ell)
print(f"  6 spectra computed, {len(ell)} multipoles each")

# Same for wCDM cosmology
print("Computing C(ell) for wCDM...")
cl_wl_wl_w = angular_cl(wcdm_dict, wl_dict, wl_dict, ell)
cl_nc_nc_w = angular_cl(wcdm_dict, nc_dict, nc_dict, ell)
cl_wl_nc_w = angular_cl(wcdm_dict, wl_dict, nc_dict, ell)
print(f"  3 spectra computed")
print(f"\nTotal: 9 tracked computations")

## 5. Inspect the stored provenance

In [ ]:
from tisserande.local_sync import execution, node, edge

all_execs = execution.get_rows()
all_nodes = node.get_rows()
all_edges = edge.get_rows()

print(f"Provenance summary:")
print(f"  Executions: {len(all_execs)}")
print(f"  Nodes:      {len(all_nodes)}")
print(f"  Edges:      {len(all_edges)}")

from collections import Counter
type_counts = Counter(str(n.type_) for n in all_nodes)
print(f"\nNode types:")
for t, c in sorted(type_counts.items()):
    print(f"  {t}: {c}")

In [ ]:
# Inspect one execution
ex = all_execs[0]
print(f"Execution {str(ex.id_)[:12]}...")
print(f"  Status: {ex.status}, Duration: {ex.duration_seconds:.3f}s")

exec_nodes = node.find_by(execution_id=ex.id_)
for n in exec_nodes:
    label = f"  {n.type_}"
    if n.arg_name:
        label += f" (arg_name={n.arg_name})"
    if hasattr(n, 'config_data') and n.config_data:
        if 'tracer_type' in n.config_data:
            label += f" [tracer: {n.config_data['tracer_type']}]"
        elif 'Omega_c' in n.config_data:
            label += f" [cosmo: Omega_c={n.config_data['Omega_c']:.3f}]"
    if hasattr(n, 'value_json') and isinstance(n.value_json, list):
        label += f" [{len(n.value_json)} values]"
    print(label)

## 6. Visualize the angular power spectra

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

prefactor = ell * (ell + 1) / (2 * np.pi)
prefactor = 1

specs = [
    ("WL x WL", cl_wl_wl, cl_wl_wl_w),
    ("NC x NC", cl_nc_nc, cl_nc_nc_w),
    ("WL x NC", cl_wl_nc, cl_wl_nc_w),
    ("CMB x CMB", cl_cmb_cmb, None),
    ("WL x CMB", cl_wl_cmb, None),
    ("NC x CMB", cl_nc_cmb, None),
]

for ax, (title, cl_planck, cl_wcdm) in zip(axes.flat, specs):
    ax.loglog(ell, np.abs(prefactor * cl_planck), "b-", lw=1.5, label="Planck")
    if cl_wcdm is not None:
        ax.loglog(ell, np.abs(prefactor * cl_wcdm), "r--", lw=1.5, label="wCDM")
    ax.set_xlabel(r"$\ell$")
    ax.set_ylabel(r"$\ell(\ell+1) C_\ell / 2\pi$")
    ax.set_title(title)
    ax.legend(fontsize=8)

fig.suptitle("Angular Power Spectra from Stored Tracers", fontsize=14)
plt.tight_layout()
plt.show()

## 7. Ratio plot: sensitivity to cosmology

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (title, cl_p, cl_w) in zip(axes, [
    ("WL x WL", cl_wl_wl, cl_wl_wl_w),
    ("NC x NC", cl_nc_nc, cl_nc_nc_w),
    ("WL x NC", cl_wl_nc, cl_wl_nc_w),
]):
    ratio = cl_w / cl_p
    ax.semilogx(ell, ratio, "k-", lw=1.5)
    ax.axhline(1.0, color="gray", linestyle="--", lw=0.5)
    ax.set_xlabel(r"$\ell$")
    ax.set_ylabel(r"$C_\ell^{\rm wCDM} / C_\ell^{\rm Planck}$")
    ax.set_title(title)
    ax.set_ylim(0.5, 1.5)

fig.suptitle("wCDM / Planck ratio — sensitivity to dark energy", fontsize=12)
plt.tight_layout()
plt.show()

## 8. What's in the database?

Let's summarize everything we stored.

## 8. Latent space encoding of C(ℓ)

Compress the 1990-point C(ℓ) curves into a low-dimensional latent
representation via PCA. This is the first step toward building an
emulator that maps cosmological parameters → latent code → C(ℓ).

In [ ]:
from tissage_cosmique.emulators import PCACodec

# Collect all Planck C(ell) curves into a matrix
# Work in log-space since C(ell) spans orders of magnitude
cl_curves = np.array([
    np.log10(np.abs(cl_wl_wl)),
    np.log10(np.abs(cl_nc_nc)),
    np.log10(np.abs(cl_wl_nc) + 1e-30),
    np.log10(np.abs(cl_cmb_cmb)),
    np.log10(np.abs(cl_wl_cmb) + 1e-30),
    np.log10(np.abs(cl_nc_cmb) + 1e-30),
    np.log10(np.abs(cl_wl_wl_w)),
    np.log10(np.abs(cl_nc_nc_w)),
    np.log10(np.abs(cl_wl_nc_w) + 1e-30),
])
print(f"C(ell) curve matrix: {cl_curves.shape}  (9 spectra x {len(ell)} multipoles)")

# Fit PCA codec
codec = PCACodec(n_components=5)
codec.fit(cl_curves)

print(f"\nPCA explained variance per component:")
for i, ev in enumerate(codec.explained_variance_ratio):
    print(f"  PC{i+1}: {ev:.6f} ({ev*100:.3f}%)")
print(f"  Total: {sum(codec.explained_variance_ratio):.8f}")

In [ ]:
# Encode all curves into latent space
Z = codec.encode(cl_curves)
print(f"Latent codes: {Z.shape}  (9 spectra x 5 components)")

labels = [
    "WL×WL (P)", "NC×NC (P)", "WL×NC (P)",
    "CMB×CMB (P)", "WL×CMB (P)", "NC×CMB (P)",
    "WL×WL (w)", "NC×NC (w)", "WL×NC (w)",
]

print(f"\nLatent codes per spectrum:")
for label, z in zip(labels, Z):
    code_str = ", ".join(f"{v:+.3f}" for v in z)
    print(f"  {label:14s}  [{code_str}]")

In [ ]:
# Decode back and check reconstruction quality
cl_reconstructed = codec.decode(Z)

fig, axes = plt.subplots(3, 3, figsize=(15, 12))

for idx, (ax, label) in enumerate(zip(axes.flat, labels)):
    original = cl_curves[idx]
    recon = cl_reconstructed[idx]
    ax.plot(ell, original, "b-", lw=1.5, label="Original")
    ax.plot(ell, recon, "r--", lw=1, label="Reconstructed")
    ax.set_xlabel(r"$\ell$")
    ax.set_ylabel(r"$\log_{10} |C_\ell|$")
    ax.set_title(label)
    if idx == 0:
        ax.legend(fontsize=8)

fig.suptitle("PCA reconstruction of C(ℓ) from 5 latent components", fontsize=13)
plt.tight_layout()
plt.show()

# Reconstruction error
rel_err = np.abs(cl_reconstructed - cl_curves) / np.maximum(np.abs(cl_curves), 1e-10)
print(f"Reconstruction error (log-space):")
print(f"  Mean: {np.mean(rel_err):.6f}")
print(f"  Max:  {np.max(rel_err):.6f}")

## 9. What's in the database?

Let's summarize everything we stored.

In [ ]:
all_cosmo = cosmology_params.get_rows()
all_tracers = tracer_config.get_rows()

print("=== Database contents ===")
print(f"\nCosmologyParams: {len(all_cosmo)} records")
for c in all_cosmo:
    print(f"  {c.name}: Omega_c={c.Omega_c}, h={c.h}, w0={c.w0}")

print(f"\nTracerConfig: {len(all_tracers)} records")
for t in all_tracers:
    extra = ""
    if t.tracer_type == "cmb_lensing":
        extra = f", z_source={t.z_source}"
    elif t.dndz_z:
        extra = f", {len(t.dndz_z)} z-points"
    print(f"  {t.name}: {t.tracer_type}{extra}")

print(f"\nProvenance: {len(all_execs)} executions, {len(all_nodes)} nodes, {len(all_edges)} edges")
print(f"\nEach angular_cl execution stores:")
print(f"  - cosmo_params (config_dict node)")
print(f"  - tracer1_config (config_dict node)")
print(f"  - tracer2_config (config_dict node)")
print(f"  - ell array (array node)")
print(f"  - C(ell) result (array node)")
print(f"  - function node + 5 edges")

In [ ]:
# Cleanup
import os
from tisserande.tracking.decorator import reset
reset()
asyncio.run(close_db())
if os.path.exists("angular_cl_demo.db"):
    os.remove("angular_cl_demo.db")
    print("Cleaned up angular_cl_demo.db")

## Summary

This notebook demonstrated the full angular C(ℓ) pipeline:

```
CosmologyParams (macon CRUD) ─┐
TracerConfig (macon CRUD) ────┤→ angular_cl (@track provenance)
TracerConfig (macon CRUD) ────┘    → C(ℓ) array
                                   → execution/node/edge records (tisserande)
```

Key points:
- **Tracer configurations** are stored independently of cosmology — reusable across different parameter sets
- **Provenance** captures all inputs (cosmology + both tracers + ℓ values) and the output C(ℓ)
- **Multiple cosmologies** with the same tracers show sensitivity to dark energy
- All data is queryable via the CLI, FastAPI, or dashboard